In [1]:
#Importando as Bibliotecas utilizadas

In [2]:
import geopandas as gpd
import pandas as pd
import os
caminho_shapefiles = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds'
caminho_escolas = os.path.join(caminho_shapefiles, 'REL_ESCOLAS_MAT.shp')

gdf_escolas_teste = gpd.read_file(caminho_escolas)
print(f" Arquivo carregado: {len(gdf_escolas_teste)} escolas")
print(f" CRS: {gdf_escolas_teste.crs}")
print(" Colunas disponíveis no arquivo:")
for col in gdf_escolas_teste.columns:
    print(f"{col}")

 Arquivo carregado: 84 escolas
 CRS: EPSG:31983
 Colunas disponíveis no arquivo:
NOME_ENTID
CD_ESCOLA
MAT_UND
M_EDINF
CRECHE
PRE_ESC
ENS_FUND
FUND_AI
FUND_AF
geometry


In [4]:
# Integração Espacial
caminho_shapefiles = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds'
caminho_apond = os.path.join(caminho_shapefiles, 'apond_timon.shp')
caminho_escolas = os.path.join(caminho_shapefiles, 'REL_ESCOLAS_MAT.shp')
caminho_saida = os.path.join(caminho_shapefiles, 'Escolas_por_Apond_Timon_Limpo.gpkg')
AP_EXCLUIR = '2112209008'
ESCOLA_EXCLUIR = '21164495'
# 1. Carregar Layers
gdf_apond = gpd.read_file(caminho_apond)
gdf_escolas = gpd.read_file(caminho_escolas)
print(f"APs: {len(gdf_apond)} | Escolas: {len(gdf_escolas)}")
# 2. Filtrar APs VÁLIDAS
gdf_apond_validas = gdf_apond[gdf_apond['cd_apond'] != AP_EXCLUIR].copy()
print(f"AP {AP_EXCLUIR} excluída. Restam {len(gdf_apond_validas)} APs para junção.")

# 3. Filtrar Escola removida
col_id = 'CD_ESCOLA' if 'CD_ESCOLA' in gdf_escolas.columns else 'CO_ENTIDADE'
if col_id in gdf_escolas.columns:
    gdf_escolas_filtradas = gdf_escolas[gdf_escolas[col_id].astype(str) != ESCOLA_EXCLUIR].copy()
    escolas_removidas = len(gdf_escolas) - len(gdf_escolas_filtradas)
    print(f"Escola {ESCOLA_EXCLUIR} excluída ({escolas_removidas} registro(s) removido(s)).")
else:
    gdf_escolas_filtradas = gdf_escolas.copy()
    print(f"Coluna de ID não encontrada. Pulando exclusão por ID.")

# 4. Padronizar SRC/CRS e Mesclar
gdf_escolas_filtradas = gdf_escolas_filtradas.to_crs(gdf_apond_validas.crs)
escolas_com_ap = gpd.sjoin(
    gdf_escolas_filtradas, 
    gdf_apond_validas[['cd_apond', 'geometry']], 
    how='left', 
    predicate='intersects'
)
# Remover escolas que caíram fora das APs válidas (ex: a escola da AP 008)
escolas_validas = escolas_com_ap.dropna(subset=['cd_apond']).copy()
print(f"{len(escolas_validas)} escolas atribuídas com sucesso às 9 APs válidas.")

# 5. Agregação por AP
colunas_matriculas = ['MAT_UND', 'M_EDINF', 'CRECHE', 'PRE_ESC', 'ENS_FUND', 'FUND_AI', 'FUND_AF']
colunas_existentes = [col for col in colunas_matriculas if col in escolas_validas.columns]
# Dicionário de agregação
agg_dict = {}
# Contar escolas
col_nome = 'NOME_ENTID' if 'NOME_ENTID' in escolas_validas.columns else 'NO_ENTIDADE'
if col_id in escolas_validas.columns:
    agg_dict[col_id] = 'count'
elif col_nome in escolas_validas.columns:
    agg_dict[col_nome] = 'count'

# Somar matrículas
for col in colunas_existentes:
    agg_dict[col] = 'sum'

# Concatenar nomes das escolas
if col_nome in escolas_validas.columns:
    agg_dict[col_nome] = lambda x: ' | '.join(x.astype(str))

# Executar agregação
agg_escolas = escolas_validas.groupby('cd_apond').agg(agg_dict).reset_index()

# Renomear coluna de contagem para 'QTD_ESCOLAS'
if col_id in agg_escolas.columns:
    agg_escolas = agg_escolas.rename(columns={col_id: 'QTD_ESCOLAS'})
elif f"{col_nome}_count" in agg_escolas.columns:
    agg_escolas = agg_escolas.rename(columns={f"{col_nome}_count": 'QTD_ESCOLAS'})

# Preencher APs sem escolas com 0
agg_escolas = agg_escolas.fillna(0)
if col_nome in agg_escolas.columns:
    agg_escolas[col_nome] = agg_escolas[col_nome].replace({0: 'Sem escolas', '0.0': 'Sem escolas'})

print(f"Agregação concluída: {len(agg_escolas)} APs processadas.")

# 6. VALIDAÇÃO
if 'MAT_UND' in agg_escolas.columns:
    total_original = gdf_escolas_filtradas['MAT_UND'].sum()
    total_agregado = agg_escolas['MAT_UND'].sum()
    print(f"Total matrículas (antes do join): {total_original:,.0f}")
    print(f"Total matrículas (após agregação): {total_agregado:,.0f}")
    print(f"Diferença: {total_original - total_agregado:,.0f}")
    
    if abs(total_original - total_agregado) < 1:
        print("Validação OK: Totais coincidem perfeitamente!")
    else:
        print("Atenção: Diferença detectada (possivelmente escolas fora dos polígonos).")

# 7. MERGE COM AS APs PARA RECUPERAR A GEOMETRIA
# Fazer merge dos dados agregados com o shapefile das APs válidas
gdf_final_escolas = gdf_apond_validas.merge(agg_escolas, on='cd_apond', how='left')

# Preencher APs sem escolas
colunas_numericas = ['QTD_ESCOLAS', 'MAT_UND', 'M_EDINF', 'CRECHE', 'PRE_ESC', 'ENS_FUND', 'FUND_AI', 'FUND_AF']
for col in colunas_numericas:
    if col in gdf_final_escolas.columns:
        gdf_final_escolas[col] = gdf_final_escolas[col].fillna(0)

if col_nome in gdf_final_escolas.columns:
    gdf_final_escolas[col_nome] = gdf_final_escolas[col_nome].fillna('Sem escolas')

print(f"GeoDataFrame final com {len(gdf_final_escolas)} APs e geometria preservada.")

# Salvar GeoPackage
gdf_final_escolas.to_file(caminho_saida, driver='GPKG', encoding='utf-8')
print(f"Arquivo salvo com sucesso: {caminho_saida}")

# Salvar também CSV para consulta rápida
caminho_csv = caminho_saida.replace('.gpkg', '.csv')
gdf_final_escolas.drop(columns=['geometry']).to_csv(caminho_csv, index=False, sep=';', decimal=',', encoding='utf-8')
print(f"CSV de consulta salvo: {caminho_csv}")

APs: 10 | Escolas: 84
AP 2112209008 excluída. Restam 9 APs para junção.
Escola 21164495 excluída (0 registro(s) removido(s)).
83 escolas atribuídas com sucesso às 9 APs válidas.
Agregação concluída: 9 APs processadas.
Total matrículas (antes do join): 31,933
Total matrículas (após agregação): 31,004
Diferença: 929
Atenção: Diferença detectada (possivelmente escolas fora dos polígonos).
GeoDataFrame final com 9 APs e geometria preservada.
Arquivo salvo com sucesso: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Escolas_por_Apond_Timon_Limpo.gpkg
CSV de consulta salvo: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Escolas_por_Apond_Timon_Limpo.csv


In [6]:
#Debug
import geopandas as gpd
caminho_escolas = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\REL_ESCOLAS_MAT.shp'
gdf = gpd.read_file(caminho_escolas)
print("Colunas do arquivo de escolas:")
for col in gdf.columns:
    print(f"  - {col}")
print("Amostra da coluna de ID (primeiros 5 valores):")
#identificar qual coluna tem o código 21164495
for col in gdf.columns:
    if gdf[col].dtype == 'object' or gdf[col].dtype == 'int64':
        if '21164495' in gdf[col].astype(str).values:
            print(f"Coluna '{col}' contém o código 21164495!")
            print(f"Primeiros valores: {gdf[col].head().tolist()}")

Colunas do arquivo de escolas:
  - NOME_ENTID
  - CD_ESCOLA
  - MAT_UND
  - M_EDINF
  - CRECHE
  - PRE_ESC
  - ENS_FUND
  - FUND_AI
  - FUND_AF
  - geometry
Amostra da coluna de ID (primeiros 5 valores):


In [7]:
caminho_arquivo_gerado = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Escolas_por_Apond_Timon_Limpo.gpkg'
try:
    # Ler o arquivo que já foi gerado
    gdf_gerado = gpd.read_file(caminho_arquivo_gerado)
    print(f"Arquivo lido com sucesso. Total de APs: {len(gdf_gerado)}\n")
    # 1. VERIFICAR A ESCOLA 21164495
    # Usamos .astype(str) e .contains para pegar '21164495', '21164495.0', etc.
    verifica_exclusao = gdf_gerado[gdf_gerado['CD_ESCOLA'].astype(str).str.contains('21164495', na=False)]
    
    if len(verifica_exclusao) > 0:
        print("ATENÇÃO: A escola 21164495 AINDA ESTÁ NO ARQUIVO!")
        print("Detalhes encontrados:")
        print(verifica_exclusao[['cd_apond', 'CD_ESCOLA', 'NOME_ENTID', 'MAT_UND']])
    else:
        print("SUCESSO: A escola 21164495 NÃO foi encontrada (foi excluída corretamente).")

    print("-" * 60)

    # 2. VERIFICAR SE A ESCOLA ALUÍZIO PERMANECEU (como você solicitou)
    verifica_aluizio = gdf_gerado[gdf_gerado['NOME_ENTID'].str.contains('ALUIZIO', case=False, na=False)]
    
    if len(verifica_aluizio) > 0:
        print("SUCESSO: A escola 'ALUIZIO DE AZEVEDO' PERMANECEU no arquivo.")
        print("Detalhes encontrados:")
        print(verifica_aluizio[['cd_apond', 'CD_ESCOLA', 'NOME_ENTID', 'MAT_UND']])
    else:
        print("ATENÇÃO: A escola 'ALUIZIO' foi excluída por engano!")

    print("=" * 60)

except FileNotFoundError:
    print(f"Erro: O arquivo não foi encontrado no caminho:\n{caminho_arquivo_gerado}")
except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo: {e}")

Arquivo lido com sucesso. Total de APs: 9

Ocorreu um erro ao ler o arquivo: 'CD_ESCOLA'


In [8]:
caminho_original = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\REL_ESCOLAS_MAT.shp'
caminho_gerado = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Escolas_por_Apond_Timon_Limpo.gpkg'
# 1. Ler o arquivo original para descobrir o NOME da escola 21164495
gdf_orig = gpd.read_file(caminho_original)
# Normalizar o ID para string sem '.0' para garantir a busca exata
gdf_orig['ID_LIMPO'] = gdf_orig['CD_ESCOLA'].astype(str).str.replace('.0', '', regex=False).str.strip()
escola_alvo = gdf_orig[gdf_orig['ID_LIMPO'] == '21164495']

if len(escola_alvo) > 0:
    nome_escola_excluir = escola_alvo.iloc[0]['NOME_ENTID']
    print(f"Escola a ser excluída identificada no original:")
    print(f"ID: 21164495")
    print(f"NOME: {nome_escola_excluir}")
else:
    print("A escola com ID 21164495 NÃO existe no arquivo original!")
    print("Amostra de IDs no arquivo:", gdf_orig['CD_ESCOLA'].head().tolist())
    nome_escola_excluir = "DESCONHECIDA"

# 2. Verificar no arquivo GERADO (agregado)
try:
    gdf_gerado = gpd.read_file(caminho_gerado)
    print(f"Arquivo gerado lido com sucesso. Total de APs: {len(gdf_gerado)}\n")
    # Como a coluna CD_ESCOLA foi agregada, vamos procurar pelo NOME da escola
    # na coluna de textos concatenados (NOME_ENTID)
    if nome_escola_excluir != "DESCONHECIDA":
        # Pegamos a primeira palavra do nome para fazer uma busca segura no texto concatenado
        palavra_chave = nome_escola_excluir.split()[0] 
        verifica_nome = gdf_gerado[gdf_gerado['NOME_ENTID'].str.contains(palavra_chave, case=False, na=False)]
        if len(verifica_nome) > 0:
            print(f"ATENÇÃO: A escola '{nome_escola_excluir}' AINDA APARECE no arquivo gerado!")
            print("Está presente nas seguintes APs:")
            print(verifica_nome[['cd_apond', 'QTD_ESCOLAS', 'NOME_ENTID']])
        else:
            print(f"SUCESSO ABSOLUTO: A escola '{nome_escola_excluir}' NÃO foi encontrada no arquivo gerado.")
            print(" A exclusão funcionou perfeitamente!")
    else:
        print("Não foi possível verificar no arquivo gerado pois o ID original não foi encontrado.")
        
except Exception as e:
    print(f"Erro ao ler o arquivo gerado: {e}")

Escola a ser excluída identificada no original:
ID: 21164495
NOME: CENTRO DE ENSINO MARIA DA CONCEICAO TEOFILO SILVA
Arquivo gerado lido com sucesso. Total de APs: 9

ATENÇÃO: A escola 'CENTRO DE ENSINO MARIA DA CONCEICAO TEOFILO SILVA' AINDA APARECE no arquivo gerado!
Está presente nas seguintes APs:
     cd_apond  QTD_ESCOLAS                                         NOME_ENTID
0  2112209009            6  CENTRO EDUCA MAIS ROBSON PARENTES NOLETO SILVA...
2  2112209003           10  CENTRO EDUCA MAIS ANNA BERNARDES | EMEF EDGARD...
4  2112209004           13  CENTRO EDUCA MAIS DEPUTADO JOSE RIBAMAR ELOUF ...
6  2112209007           10  CENTRO EDUCA MAIS JACIRA DE OLIVEIRA E SILVA |...
7  2112209005            9  CENTRO DE ENSINO ALUIZIO DE AZEVEDO | CENTRO D...
8  2112209002           12  CENTRO DE ENSINO DR JOAO LULA | EMEF ANTONIA D...


In [9]:
# Exclusão AP 008 + Escola 21164495
import geopandas as gpd
import pandas as pd
import os
# Configurações
caminho_shapefiles = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds'
caminho_apond = os.path.join(caminho_shapefiles, 'apond_timon.shp')
caminho_escolas = os.path.join(caminho_shapefiles, 'REL_ESCOLAS_MAT.shp')
caminho_saida = os.path.join(caminho_shapefiles, 'Escolas_por_Apond_Timon_Limpo_Final.gpkg')
AP_EXCLUIR = '2112209008'
ESCOLA_ID_EXCLUIR = '21164495'
ESCOLA_NOME_EXCLUIR = 'CENTRO DE ENSINO MARIA DA CONCEICAO TEOFILO SILVA'
# 1. CARREGAR CAMADAS
gdf_apond = gpd.read_file(caminho_apond)
gdf_escolas = gpd.read_file(caminho_escolas)
print(f"APs: {len(gdf_apond)} | Escolas: {len(gdf_escolas)}")

# 2. FILTRAR APs VÁLIDAS (Exclusão Precoce da AP 008)
gdf_apond_validas = gdf_apond[gdf_apond['cd_apond'] != AP_EXCLUIR].copy()
print(f"AP {AP_EXCLUIR} excluída. Restam {len(gdf_apond_validas)} APs.")

# 3. FILTRAR ESCOLA POR ID E NOME (DUPLO FILTRO)
# Normalizar ID para string
gdf_escolas['ID_LIMPO'] = gdf_escolas['CD_ESCOLA'].astype(str).str.replace('.0', '', regex=False).str.strip()
# Filtro por ID
filtro_id = gdf_escolas['ID_LIMPO'] != ESCOLA_ID_EXCLUIR
# Filtro por nome (case-insensitive)
filtro_nome = ~gdf_escolas['NOME_ENTID'].str.contains('MARIA DA CONCEICAO TEOFILO', case=False, na=False)
# Aplicar ambos os filtros
gdf_escolas_filtradas = gdf_escolas[filtro_id & filtro_nome].copy()
escolas_removidas = len(gdf_escolas) - len(gdf_escolas_filtradas)
print(f"Escola(s) excluída(s): {escolas_removidas} registro(s) removido(s).")
# Remover coluna temporária
gdf_escolas_filtradas = gdf_escolas_filtradas.drop(columns=['ID_LIMPO'])
# 4. PADRONIZAR CRS E SPATIAL JOIN
gdf_escolas_filtradas = gdf_escolas_filtradas.to_crs(gdf_apond_validas.crs)
escolas_com_ap = gpd.sjoin(
    gdf_escolas_filtradas, 
    gdf_apond_validas[['cd_apond', 'geometry']], 
    how='left', 
    predicate='intersects'
)
# Remover escolas que caíram fora das APs válidas
escolas_validas = escolas_com_ap.dropna(subset=['cd_apond']).copy()
print(f"{len(escolas_validas)} escolas atribuídas às APs válidas.")

# 5. AGREGAÇÃO POR ÁREA DE PONDERAÇÃO
colunas_matriculas = ['MAT_UND', 'M_EDINF', 'CRECHE', 'PRE_ESC', 'ENS_FUND', 'FUND_AI', 'FUND_AF']
colunas_existentes = [col for col in colunas_matriculas if col in escolas_validas.columns]
agg_dict = {
    'CD_ESCOLA': 'count',
    'NOME_ENTID': lambda x: ' | '.join(x.astype(str))
}
for col in colunas_existentes:
    agg_dict[col] = 'sum'

agg_escolas = escolas_validas.groupby('cd_apond').agg(agg_dict).reset_index()
agg_escolas = agg_escolas.rename(columns={'CD_ESCOLA': 'QTD_ESCOLAS'})

# Preencher APs sem escolas
agg_escolas = agg_escolas.fillna(0)
agg_escolas['NOME_ENTID'] = agg_escolas['NOME_ENTID'].replace({0: 'Sem escolas', '0.0': 'Sem escolas'})

print(f"Agregação concluída: {len(agg_escolas)} APs processadas.")

# 6. MERGE COM GEOMETRIA DAS APs
gdf_final_escolas = gdf_apond_validas.merge(agg_escolas, on='cd_apond', how='left')
colunas_numericas = ['QTD_ESCOLAS', 'MAT_UND', 'M_EDINF', 'CRECHE', 'PRE_ESC', 'ENS_FUND', 'FUND_AI', 'FUND_AF']
for col in colunas_numericas:
    if col in gdf_final_escolas.columns:
        gdf_final_escolas[col] = gdf_final_escolas[col].fillna(0)

gdf_final_escolas['NOME_ENTID'] = gdf_final_escolas['NOME_ENTID'].fillna('Sem escolas')

print(f"GeoDataFrame final: {len(gdf_final_escolas)} APs com geometria.")

# 7. VALIDAÇÃO FINAL E EXPORTAÇÃO
# Verificar se a escola foi excluída
verifica_exclusao = gdf_final_escolas[gdf_final_escolas['NOME_ENTID'].str.contains('MARIA DA CONCEICAO TEOFILO', case=False, na=False)]
if len(verifica_exclusao) == 0:
    print("SUCESSO: Escola 21164495 NÃO está no arquivo final!")
else:
    print("ATENÇÃO: Escola ainda presente!")
# Validar totais
if 'MAT_UND' in gdf_final_escolas.columns:
    total_original = gdf_escolas_filtradas['MAT_UND'].sum()
    total_final = gdf_final_escolas['MAT_UND'].sum()
    print(f"Total matrículas (filtrado): {total_original:,.0f}")
    print(f"Total matrículas (final): {total_final:,.0f}")
    print(f"Diferença: {total_original - total_final:,.0f}")
# Exportar
gdf_final_escolas.to_file(caminho_saida, driver='GPKG', encoding='utf-8')
print(f"GeoPackage salvo: {caminho_saida}")
caminho_csv = caminho_saida.replace('.gpkg', '.csv')
gdf_final_escolas.drop(columns=['geometry']).to_csv(caminho_csv, index=False, sep=';', decimal=',', encoding='utf-8')
print(f"CSV salvo: {caminho_csv}")

APs: 10 | Escolas: 84
AP 2112209008 excluída. Restam 9 APs.
Escola(s) excluída(s): 1 registro(s) removido(s).
83 escolas atribuídas às APs válidas.
Agregação concluída: 9 APs processadas.
GeoDataFrame final: 9 APs com geometria.
SUCESSO: Escola 21164495 NÃO está no arquivo final!
Total matrículas (filtrado): 31,004
Total matrículas (final): 31,004
Diferença: 0
GeoPackage salvo: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Escolas_por_Apond_Timon_Limpo_Final.gpkg
CSV salvo: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Escolas_por_Apond_Timon_Limpo_Final.csv
